<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Common Agent Failure Cases</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>


# Demo: Common Agent Failure Cases in IT Support

**Purpose:**  
This notebook demonstrates typical ways AI agent systems fail in real environments and how to identify them through execution traces.

**Case Scenario:**  
In a company's IT department, agents are used to triage support tickets, diagnose issues, and recommend resolutions. Failures in these agents can lead to unresolved tickets, wasted engineer time, poor employee experience, and delayed escalations for critical issues.

We will simulate an **IT Helpdesk Agent** that monitors support tickets, checks resolution history, and predicts escalation risk.

**Acknowlodegment**
 The code and structure of this lab have been adapted and reused from materials developed by WeCloudData.


## Dataset Description

The dataset used in this lab is a **simulated IT Helpdesk dataset**. It is not loaded from an external file but is instead embedded directly within the custom tool functions:

*   `check_ticket_status`: Contains current status, priority, and assignment information for tickets like TCK-101, TCK-204, and TCK-317.
*   `get_resolution_history`: Provides historical information, such as how many times a ticket has been reopened or previous issues.
*   `predict_escalation_risk`: Offers risk assessments indicating whether a ticket might require higher-level support.

This simulated dataset was specifically crafted to create scenarios that expose common AI agent failure cases (hallucination, tool misuse, ignoring instructions) during the agent's interaction with these tools.

## 1. Setup

In [1]:
# Setup - Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 422.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/

## 2. Tools for IT Helpdesk Agent

These tools simulate real IT service desk data sources.

In [2]:
from langchain_core.tools import tool

@tool
def check_ticket_status(ticket_id: str) -> str:
    """Check the current status of a support ticket."""

    status_db = {
        "TCK-101": "High priority. User cannot connect to VPN. Not assigned yet.",
        "TCK-204": "Low priority. Printer is not working. Assigned to Desktop Support.",
        "TCK-317": "Medium priority. Outlook does not open. Waiting for the user."
    }

    return status_db.get(ticket_id, f"No information for ticket {ticket_id}.")


@tool
def get_resolution_history(ticket_id: str) -> str:
    """Check what was tried before for this ticket."""

    history = {
        "TCK-101": "Opened again 3 times this month. The same VPN error happened each time.",
        "TCK-204": "First time this happened. No previous issues with this device.",
        "TCK-317": "Outlook was reinstalled 5 days ago. The problem came back after 24 hours."
    }

    return history.get(ticket_id, "No previous history for this ticket.")


@tool
def predict_escalation_risk(ticket_id: str) -> str:
    """Check if this ticket may need help from a higher support team."""

    risk_db = {
        "TCK-101": "High risk. The VPN problem keeps happening.",
        "TCK-204": "Low risk. A normal printer fix should solve the problem.",
        "TCK-317": "Medium risk. The problem keeps coming back and may need extra support."
    }

    return risk_db.get(ticket_id, "No risk information available.")

## 3. IT Helpdesk Agent Setup

In [3]:
from langchain.agents import create_react_agent, AgentExecutor

In [4]:
#from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4", temperature=0)
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

tools = [check_ticket_status, get_resolution_history, predict_escalation_risk]

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print(" IT Helpdesk Agent ready for failure demonstration")

 IT Helpdesk Agent ready for failure demonstration


## 5. Failure Case 1: Hallucination

The agent invents specific facts (error codes, dates, names) that were never returned by any tool.

In [5]:
from langchain.prompts import PromptTemplate
hallucination_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

hallucination_prompt = PromptTemplate.from_template(hallucination_prompt_template)

hallucination_agent = create_react_agent(llm=llm, tools=tools, prompt=hallucination_prompt)
hallucination_executor = AgentExecutor(agent=hallucination_agent, tools=tools, verbose=True, max_iterations=6, handle_parsing_errors=True)

print("Failure Case 1: Hallucination")
response = hallucination_executor.invoke({"input": "Who fixed ticket TCK-204 and what was the exact resolution time?"})
print("\nFinal Output:", response.get("output", "No output"))

Failure Case 1: Hallucination


> Entering new AgentExecutor chain...
Thought: I need to check the resolution history to find out who fixed the ticket and what was the exact resolution time.
Action: get_resolution_history
Action Input: TCK-204First time this happened. No previous issues with this device.The observation does not provide the information I need. It seems there might be a misunderstanding in the question or the tool used. The get_resolution_history tool provides information about what was tried before for this ticket, not who resolved it or when. I cannot provide an answer based on the available tools and information.
Final Answer: I'm sorry, but I can't provide the information you're looking for.

> Finished chain.

Final Output: I'm sorry, but I can't provide the information you're looking for.


## 6. Failure Case 2: Tool Misuse

The agent calls the right tool but with a malformed or incorrect input, causing the tool to fail silently.

In [6]:
misuse_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: ...
Observation: ...
... (repeat if needed)
Thought: I now know the final answer
Final Answer: ...

Begin!

Question: {input}
{agent_scratchpad}"""

misuse_prompt = PromptTemplate.from_template(misuse_prompt_template)

tool_misuse_agent = create_react_agent(llm=llm, tools=tools, prompt=misuse_prompt)
tool_misuse_executor = AgentExecutor(agent=tool_misuse_agent, tools=tools, verbose=True, max_iterations=6, handle_parsing_errors=True)

print("Failure Case 2: Tool Misuse")
response = tool_misuse_executor.invoke({"input": "What is the escalation risk level of TCK-101?"})
print("\nFinal Output:", response.get("output", "Agent stopped"))

Failure Case 2: Tool Misuse


> Entering new AgentExecutor chain...
Thought: To find out the escalation risk level of a ticket, I need to use the predict_escalation_risk function.
Action: predict_escalation_risk
Action Input: TCK-101High risk. The VPN problem keeps happening.I now know the final answer
Final Answer: The escalation risk level of TCK-101 is high.

> Finished chain.

Final Output: The escalation risk level of TCK-101 is high.


## 7. Failure Case 3: Ignoring Instructions

The agent answers a different question than the one actually asked, ignoring the user's specific request.

In [ ]:
ignore_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: ...
Observation: ...
Thought: I now know the final answer
Final Answer: ...

Begin!

Question: {input}
{agent_scratchpad}"""

ignore_prompt = PromptTemplate.from_template(ignore_prompt_template)

ignore_instructions_agent = create_react_agent(llm=llm, tools=tools, prompt=ignore_prompt)
ignore_instructions_executor = AgentExecutor(agent=ignore_instructions_agent, tools=tools, verbose=True, max_iterations=5, handle_parsing_errors=True)

print(" Failure Case 3: Ignoring Instructions")
response = ignore_instructions_executor.invoke({"input": "Specifically, how many times has TCK-101 been reopened?"})
print("\nFinal Output:", response.get("output", "No output"))

 Failure Case 3: Ignoring Instructions


> Entering new AgentExecutor chain...
Thought: To answer this question, I need to check the resolution history of the ticket to see how many times it has been reopened.
Action: get_resolution_history
Action Input: TCK-101Opened again 3 times this month. The same VPN error happened each time.The ticket has been reopened 3 times. Now I need to check the current status of the ticket.
Action: check_ticket_status
Action Input: TCK-101High priority. User cannot connect to VPN. Not assigned yet.The ticket is of high priority and is currently unassigned. I should also check if this ticket may need help from a higher support team.
Action: predict_escalation_risk
Action Input: TCK-101High risk. The VPN problem keeps happening.I now know the final answer.
Final Answer: The ticket TCK-101 has been reopened 3 times due to a recurring VPN error. It is currently unassigned and has a high priority. There is a high risk that this ticket may need to be escalated 

## 8. Inspecting Execution Traces to Identify Issues

**How to read the trace:**

- **Thought**: What the agent is thinking
- **Action**: Which tool it chose
- **Action Input**: What parameters it used
- **Observation**: Tool result or error
- **Final Answer**: Conclusion

**Common Failure Patterns Seen:**
- Hallucination → Final Answer contains specific facts (codes, names, times) that never appeared in any Observation
- Tool misuse → Action Input format doesn't match what the tool expects, so the tool returns a "not found" / default result
- Ignoring instructions → Final Answer doesn't address what the user actually asked

In real environment, these failures can cause misleading reports, wasted engineer time, and missed critical escalations.

## Conclusion

This demo showed three common agent failure cases in an IT helpdesk context:

1. Hallucination
2. Tool misuse
3. Ignoring instructions

**Key Takeaway:**  
Carefully inspecting execution traces is the most effective way to identify and fix agent failures before deployment.